In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
from torchvision import models
import ssl

from glob import glob
from PIL import Image
import random
import os
import pandas as pd
import joblib
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from concurrent.futures import ThreadPoolExecutor
import pickle

## kiểm tra dữ liệu

In [132]:
df1 = pd.read_csv('data/csv/cicddos_2019_4_labels.csv')
df2 = pd.read_csv('temp/craw/syn_1.csv')
# df2 = pd.read_csv('temp/actual_data/test_udp.csv')

In [133]:
selected_columns1 = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']

df1 = df1[df1['Label'] == 'Syn']
df1 = df1[selected_columns1]

labels = df1['Label']

In [134]:
labels.value_counts()

Label
Syn    5000
Name: count, dtype: int64

In [135]:
df1.sample(5)
# df1['Flow IAT Mean'].sample(10)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Bwd Avg Bulk Rate,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Std,Label
17213,6,115,2,2,12.0,6.0,6.0,0.0,6.0,6.0,...,0,5840,0,20,0.0,0.000000,0.0,0.0,0.000000e+00,Syn
16749,6,1,2,0,12.0,6.0,6.0,0.0,0.0,0.0,...,0,5840,-1,20,0.0,0.000000,0.0,0.0,0.000000e+00,Syn
1371,6,121,2,2,12.0,6.0,6.0,0.0,6.0,6.0,...,0,5840,0,20,0.0,0.000000,0.0,0.0,0.000000e+00,Syn
11457,6,60155364,10,6,60.0,6.0,6.0,0.0,6.0,6.0,...,0,5840,0,20,66.5,51.797683,110.0,1.0,8.493342e+06,Syn
14883,6,1,2,0,12.0,6.0,6.0,0.0,0.0,0.0,...,0,5840,-1,20,0.0,0.000000,0.0,0.0,0.000000e+00,Syn


In [136]:
selected_columns2 = ['Source IP','Dest IP','Protocol','Flow Duration', 'Total Fwd Packets', 
                     'Total Backward Packets', 'Fwd Packets Length Total', 'Fwd Packet Length Max', 
                     'Fwd Packet Length Min', 'Bwd Packets Length Total']

df2 = df2[selected_columns1]

In [138]:
df2.sample(3)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Bwd Avg Bulk Rate,Init Fwd Win Bytes,Init Bwd Win Bytes,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Std,Label
0,6,0.0002,2383,0,3186,6.0,0.0,2.496864,0.0,0.0,...,0.0,64240,0,24,0.000196,0.0,0.000196,0.000196,0.0,Syn
3,6,0.0003,136,0,168,6.0,0.0,2.426069,0.0,0.0,...,0.0,64240,0,24,0.000265,0.0,0.000265,0.000265,0.0,Syn
1,6,0.0002,688,0,792,6.0,0.0,2.362584,0.0,0.0,...,0.0,64240,0,24,0.000200,0.0,0.000200,0.000200,0.0,Syn


## chuyên dữ liệu

#### chuyển dữ liệu cách 2

In [23]:
# Load pre-trained ResNet50 model
SCALER_PATH = "scaler/scaler.pkl"
GRID_POSITIONS_PATH = "scaler/grid_positions.npy"

# Load precomputed MinMaxScaler and grid positions
scaler = joblib.load(SCALER_PATH)
grid_positions = np.load(GRID_POSITIONS_PATH)

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std']


def prepare_data(data: pd.DataFrame):
    data = data[selected_columns]
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)
    
    data = np.log1p(data + 1)
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)
    
    features = scaler.transform(data.values)  # Use precomputed scaler

    return features

def process_row_to_image(row, grid_positions, grid_size=7, image_size=224):
    grid = np.zeros((grid_size, grid_size), dtype=float)
    count = np.zeros((grid_size, grid_size), dtype=int)
    
    for i in range(len(row)):
        x, y = grid_positions[i]
        grid[x, y] += row[i]
        count[x, y] += 1
    
    nonzero = count > 0
    grid[nonzero] = grid[nonzero] / count[nonzero]
    grid_norm = ((grid - grid.min()) / (grid.max() - grid.min() + 1e-8)) * 255
    
    upscale_factor = image_size // grid_size
    expanded_image = np.kron(grid_norm, np.ones((upscale_factor, upscale_factor)))
    img = Image.fromarray(expanded_image.astype(np.uint8), mode='L').convert("RGB")
    img = img.resize((image_size, image_size))
    return img

# Load dataset
df = pd.read_csv('temp/test/test_benign.csv')
# df = df[df["Dest Post"] == 80]

# Preprocess data
features = prepare_data(df)

# Xử lý song song với 3 luồng
num_threads = 3
images = []

def process_single_row(idx):
    return process_row_to_image(features[idx], grid_positions)

with ThreadPoolExecutor(max_workers=num_threads) as executor:
    images = list(executor.map(process_single_row, range(len(features))))

i = 1
# Lưu ảnh kết quả (tùy chọn)
for i, img in enumerate(images):
    img.save(f"temp/test/image test/benign/image_{i}.png")
    i += 1
    if i == 5000:
        break

C:\Users\NewTun\AppData\Local\Temp\ipykernel_11400\403789766.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.replace([-np.inf, np.inf], 0, inplace=True)
C:\Users\NewTun\AppData\Local\Temp\ipykernel_11400\403789766.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.fillna(0, inplace=True)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


## Danh gia

In [12]:
label_map = {'BENIGN': 0, 'Group1': 1, 'Group2': 2, 'Syn': 3}

# Kiểm tra thiết bị (GPU nếu có)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print("Using device:", device)

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(label_map.keys()))  # Adjust output layer
model = model.to(device)
model.load_state_dict(torch.load("models/resnet50_finetuned.pth",  map_location=device))
print(len(label_map.keys()))

Using device: cpu
4


In [277]:
# Test individual images
def predict_image(model, image_path, class_names):
    model.eval()
    image = Image.open(image_path)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    image = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(image)
        _, predicted = torch.max(output, 1)
    return class_names[predicted.item()]



In [278]:
labels_list = list(label_map.keys())
print(labels_list)
pred = predict_image(model, "temp/data/Syn/image_1.png", list(label_map.keys()))
print(pred)

['BENIGN', 'Group1', 'Group2', 'Syn']
BENIGN


In [280]:
folder_path = "temp/data/Syn"
files = os.listdir(folder_path)  # Lấy danh sách tất cả file và thư mục

sat = {}
for i in files:
    pred = predict_image(model, f"temp/data/Syn/{i}", list(label_map.keys()))
    if pred in sat:
        sat[pred] += 1
    else:
        sat[pred] = 1

print(sat)

{'BENIGN': 50, 'Group2': 13}


## ddd

In [10]:
# Đọc file CSV
file_path = "data/csv/cicddos_2019_4_labels.csv"  # Thay bằng đường dẫn file của bạn
df = pd.read_csv(file_path)

In [11]:
# Tách dữ liệu theo từng nhãn và lưu thành file riêng
for label in df['Label'].unique():
    df_label = df[df['Label'] == label]
    output_file = f"temp/old/output_{label}.csv"
    df_label.to_csv(output_file, index=False)
    print(f"Đã lưu: {output_file}")

Đã lưu: temp/old/output_Group1.csv
Đã lưu: temp/old/output_BENIGN.csv
Đã lưu: temp/old/output_Syn.csv
Đã lưu: temp/old/output_Group2.csv


In [18]:
seleccolumns=['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
       'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio',
       'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
       'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
       'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate',
       'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets',
       'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes',
       'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
       'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max',
       'Idle Min', 'Label']

df = pd.read_csv('temp/old/dd.csv')
df = df[seleccolumns]

df.to_csv('temp/csv/new_dd.csv', index=False)

In [20]:
# Danh sách các file CSV cần hợp nhất
file_list = ["temp/old/output_BENIGN.csv", "temp/old/output_Group1.csv", "temp/old/output_Group2.csv", "temp/old/output_Syn.csv"]  # Thay tên file thực tế của bạn

# Đọc và hợp nhất dữ liệu
df_list = [pd.read_csv(file) for file in file_list]  # Đọc từng file vào danh sách
df_merged = pd.concat(df_list, ignore_index=True)  # Hợp nhất tất cả các DataFrame

# Lưu thành file CSV mới
output_file = "temp/old/1.csv"
df_merged.to_csv(output_file, index=False)
print(f"Đã lưu file hợp nhất: {output_file}")

Đã lưu file hợp nhất: temp/old/1.csv
